# Collimator Mesh Definition

Written by Lev Thomson for python 3.9.25

This script generates a mesh for the thruster and collimator using gmsh's python API. 
The mesh can then be imported into LEoPart-x / FEniCSx for use in plasma simulation. 

## Packages / Libraries:

In [1]:
import gmsh # For creating mesh
import numpy as np # For adding magnets
import meshio # For converting to xdmf format for use in fenicsx

# UP TO HERE no idea why this isn't loading

## Variables

First, variables are defined for the thruster dimensions, the plasma's Debye length and the dimensions of the mesh. 

In [2]:
run_gui = False # Show the mesh in GMSH GUI

# Thruster / collimator variables:
# TODO note that these are almost entirely dummy variables at this stage. 
# Check with ¿team mech? re: geometry of thruster.
# Do some googling to find collimator & magnet geometry

r_c = 0.0125 # m, = 12.5mm, cathode radius
r_coll = 0.05 # m, = 3cm, collimator radius
r_a = 2*r_c # m, = 25mm, anode radius
r_trig = 0.0005 # m = 0.5mm, trigger radius
kappa = 0.03 # m, = 3cm, cathode-collimator distance
l_c = r_c # m = 12.5mm, length of cathode
l_coll = 0.08 # m, = 10cm, length of collimator
l_a = l_c+kappa # m, = 42.5mm, length of anode
l_trig = l_c * 3/2 # m, = 18.75mm, length of trigger

n_mag = 3 # number of pairs of magnets TODO set range
h_mag = 0.0025 # m, = 2.5mm, height of magnet
l_mag = l_coll # m, = 10cm, length of magnet
w_mag = 2*h_mag # m, = 5mm, width of magnet

# Plasma variables:
debye_length = 0.1 # TODO dummy values for now - define as a field eventually

# Model variables:
size_min = 0.001 # TODO dummy values for now - define as a field eventually
size_max = debye_length

# File variables:

meshname = "collimator.msh" # Mesh will be saved to this file


## Functions

The number of magnets used in the collimator varies, and is specified in n_mag. The function add_magnet_pair (defined below) is called repeatedly when generating the mesh. This adds an evenly-spaced set of n_mag*2 magnets around the inner rim of the collimator. 

In [3]:
def add_magnet_pair(model:gmsh.model, x:float, y:float, z:float, dx:float, dy:float, dz:float, angle:float, last_tag:int, magnet_list:list)->gmsh.model:
    '''
    Adds a pair of magnets at a given point on the collimator-thruster mesh. 
    The magnet is a rectangle defined by a point (x', y', z') and the extents along the x-, y- and z-axes, and is rotated by angle. 
    The second magnet is added diagonally opposite the first.
        Args:
        model: the model to be updated
        x: x-coordinate of one corner of box
        y: y-coordinate of one corner of box
        z: z-coordinate of one corner of box
        dx: x-coordinate of other corner of box
        dy: y-coordinate of other corner of box
        dz: z-coordinate of other corner of box
        angle: (radians) the rectangle is rotated about the x-axis by this angle
        last_tag: the tag of the entity to add the magnets to

    Returns:
        model: updated gmsh model
        last_tag: the tag of the updated model
        magnet_list: list of magnet surfaces by [(dimension=2, tag)] (for defining field later)
    '''
    # Add first magnet
    mag_1_vol = model.occ.addBox(x,-y,-z, dx, dy, dz, tag=last_tag+1)
    model.occ.rotate([(3, mag_1_vol)], x+dx/2,0,0, 1,0,0, angle)

    # Add second magnet opposite the first
    mag_2_vol = model.occ.addBox(x,-y,-z, dx, dy, dz, tag=last_tag+2)
    model.occ.rotate([(3, mag_2_vol)], x+dx/2,0,0, 1,0,0, angle+np.pi)

    gmsh.model.occ.synchronize() # Cut both at the same time - reduces synchronise calls. 
    
    model.occ.cut([(3, last_tag)], [(3, mag_1_vol)], tag=last_tag+3, removeObject=True, removeTool=False) # Cut first magnet
    model.occ.cut([(3, last_tag+3)], [(3, mag_2_vol)], tag=last_tag+4, removeObject=True, removeTool=False) # Cut second magnet

    mag_1_surf = model.getBoundary([(3, mag_1_vol)], combined=False, oriented=False)
    mag_2_surf = model.getBoundary([(3, mag_2_vol)], combined=False, oriented=False)
    magnet_list.append(mag_1_surf)
    magnet_list.append(mag_2_surf)

    gmsh.model.occ.remove([(3, mag_1_vol)], recursive=False) # Then remove the volumes leaving the bounding surfaces intact
    gmsh.model.occ.remove([(3, mag_2_vol)], recursive=False)

    gmsh.model.occ.synchronize()

    # Update tag
    last_tag = last_tag+4
    return model, last_tag, magnet_list

## Defining a Model Using GMSH

GMSH stores data as a model, which is an object that contains topological entities. 

These entities are:
Order 0 entities: Points
Order 1 entities: Curves, which are defined by multiple points
Order 2 entities: Surfaces, which are defined by multiple curves
Order 3 entities: Volumes, which are defined by multiple surfaces

Each of these entities is identified by a dimtag, which is a tuple with values (dimension, tag).
The dimension is the order of the entity, and the tag is a positive integer that is unique to that entity within that order. (e.g., [(3,1)] is not the same as [(2,1)]. The first is a volume but the second is a surface.) Tags can be defined when the entity is created, or can be generated automatically by GMSH. 

Various operations are available that allow a user to create a complex shape from simpler entities. These include cut and intersect to remove one entity from another, fuse to join entities, and fragment and getBoundary to divide an order-x entity into its constituent parts of order x-1.  

In [4]:
# ~~~~ Set parameters ~~~~

gmsh.initialize()
gmsh.model.add("collimator") # Add model with name collimator

# ~~~~ Define geometry ~~~~

vol_coll = gmsh.model.occ.addCylinder(l_a,0,0, l_coll,0,0, r_coll, 1) # Collimator cylinder
vol_a = gmsh.model.occ.addCylinder(0,0,0, l_a,0,0, r_a,2) # Anode cylinder
vol_coll_a = gmsh.model.occ.fuse([(3, vol_coll)], [(3, vol_a)], 3)[0] # Fuse collimator to anode. Should look like 2 cylinders stacked on top of each other

# Then remove the cathode from the anode cylinder
vol_c = gmsh.model.occ.addCylinder(0,0,0, l_c,0,0, r_c, 4)
vol_trig = gmsh.model.occ.addCylinder(0,0,0, l_trig,0,0, r_trig, 5)
vol_c_trig = gmsh.model.occ.fuse([(3, vol_c)], [(3, vol_trig)], 6)[0] # Fuse cathode and trigger
vol_thruster = gmsh.model.occ.cut(vol_coll_a, vol_c_trig, 7)[0] # Cut the cathode-trigger out of the anode-collimator assembly

# Add magnets:
if n_mag == 0:
    pass
elif n_mag > 0:
    last_tag = vol_thruster[0][1] # Tag of most-recently created entity
    angle_array = np.linspace(0, np.pi, n_mag+1)[0:n_mag] # Generate array of equally spaced magnet pairs around spherical cathode surface
    magnet_list = []
    for loop_idx, angle in enumerate(angle_array):
        gmsh.model, last_tag, magnet_list = add_magnet_pair(gmsh.model, l_a, r_coll, w_mag/2, l_mag, h_mag, w_mag, angle, last_tag, magnet_list)

gmsh.model.occ.synchronize()


# Defining Fields

After the geometry of the thruster and collimator has been defined, then a field is applied to the model. This allows users to specify the mesh size programatically. For the plasma simulation, the mesh should be smallest where the magnetic field is strong. For this reason, a threshold field is applied to the mesh. This specifies a minimum mesh size, which occurs within distance DistMin to the magnet surfaces. The mesh size then increases up to a maximum size, which occurs at a distance DistMax from the magnet surfaces. 

Distmax is defined as the Debye length. This is the distance over which an ion's magnetic field has an effect, and is the largest size that a mesh cell should be in the plasma simulation. 

The mesh is then generated by GMSH. This can be a computationally intensive process, so it is done in parallel where possible. The resulting mesh is then saved in a .msh file for importing to FEniCSx. 

If the code is run on a computer that supports graphics, then it is displayed in the GMSH gui automatically. 

In [5]:
# ~~~~ Define mesh field ~~~~


# Get magnet boundary
magnet_array = np.zeros((n_mag*2*6))
for mag_idx, magnet in enumerate(magnet_list):
    for surf_idx, surface in enumerate(magnet):
        magnet_array[6*mag_idx+surf_idx] = surface[1]

# Eliminate surfaces that are shared by all magnets - this causes the mesh to be fine near the ends of the collimator
unique_surfs_vals, unique_surfs_counts = np.unique(magnet_array, return_counts=True)

for val_idx, value in enumerate(unique_surfs_vals):
    if unique_surfs_counts[val_idx] == n_mag*2: # If shared by all magnets
        magnet_array = magnet_array[magnet_array != value] # Remove all instances of that surface


gmsh.model.mesh.field.add("Distance", 1)
gmsh.model.mesh.field.setNumbers(1, "SurfacesList", magnet_array)
gmsh.model.mesh.field.setNumber(1, "Sampling", 1000)

gmsh.model.mesh.field.add("Threshold", 2)
gmsh.model.mesh.field.setNumber(2, "InField", 1) # Create a mesh field that defines mesh size based on proximity to magnets
gmsh.model.mesh.field.setNumber(2, "SizeMin", size_min)
gmsh.model.mesh.field.setNumber(2, "SizeMax", size_max)
gmsh.model.mesh.field.setNumber(2, "DistMin", 0)
gmsh.model.mesh.field.setNumber(2, "DistMax", 0.0005) # 0.5mm
gmsh.model.mesh.field.setAsBackgroundMesh(2)

for surf in magnet_array:
    gmsh.model.occ.remove([(2, surf)], recursive=False)

# ~~~~ Save & output to GUI ~~~~

gmsh.model.occ.synchronize()

# ~~~~ Parallelize ~~~~

gmsh.option.setNumber("General.NumThreads", 5) # Use multithreading to generate mesh faster
gmsh.option.setNumber("Mesh.Algorithm3D", 4) # Frontal-Delaunay (default. If this causes issues consider changing to 10 - HXT which is a version of Delaunay that is optimised for parallel meshing). 

# ~~~~ Mesh ~~~~

gmsh.model.mesh.generate(3) # Generate in 3d

gmsh.write(meshname)

if run_gui: # Off by default for remote server. 
    gmsh.fltk.run()

gmsh.finalize()

print("Done")

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 13 (Line)
Info    : [  0%] Meshing curve 16 (Line)
Info    : [  0%] Meshing curve 14 (Line)
Info    : [  0%] Meshing curve 15 (Line)
Info    : [  0%] Meshing curve 17 (Line)
Info    : [  0%] Meshing curve 18 (Line)
Info    : [  0%] Meshing curve 19 (Line)
Info    : [  0%] Meshing curve 20 (Line)
Info    : [  0%] Meshing curve 21 (Line)
Info    : [ 10%] Meshing curve 22 (Line)
Info    : [ 10%] Meshing curve 23 (Line)
Info    : [ 10%] Meshing curve 24 (Line)
Info    : [ 10%] Meshing curve 25 (Line)
Info    : [ 10%] Meshing curve 26 (Line)
Info    : [ 10%] Meshing curve 27 (Line)
Info    : [ 10%] Meshing curve 28 (Line)
Info    : [ 10%] Meshing curve 29 (Line)
Info    : [ 10%] Meshing curve 30 (Line)
Info    : [ 20%] Meshing curve 31 (Line)
Info    : [ 20%] Meshing curve 32 (Line)
Info    : [ 20%] Meshing curve 33 (Line)
Info    : [ 20%] Meshing curve 34 (Line)
Info    : [ 20%] Meshing curve 35 (Line)
Info    : [ 20%] Meshing curve 36

In [ ]:
msh = meshio.read(meshname)

# UP TO HERE need to read up on how exactly all of this works. I think I need to define a physical group in the mesh?

